# NEXUS — Supply Chain Disruption Response Coordinator

Q6 runnable demo notebook. This notebook executes the same five-agent blackboard pipeline used by the FastAPI service and prints the trace plus final decision.

In [ ]:
from pathlib import Path
import sys
import json

# Works when launched from either the project root or the backend folder.
cwd = Path.cwd()
backend_dir = cwd if (cwd / 'app').exists() else cwd / 'backend'
if not (backend_dir / 'app').exists():
    raise FileNotFoundError('Open this notebook from the project root or backend folder.')
sys.path.insert(0, str(backend_dir))

from app.data.repository import get_scenarios
from app.engine.orchestrator import run_scenario

print('Backend:', backend_dir)

## 1. Select a disruption scenario

The sample records are fictional and stored in `app/data/csv/`.

In [ ]:
scenarios = get_scenarios()
for index, scenario in enumerate(scenarios, start=1):
    print(f"{index}. {scenario['title']} | {scenario['scenario_id']} | {scenario['required_qty']:,} units")

selected = dict(scenarios[0])  # Change 0 to 1 or 2 to try another scenario.
print('\nSelected:', selected['description'])

## 2. Run the five agents

Each specialist calls two ordinary Python tools, reads the shared Blackboard when its inputs are available, and posts a structured finding.

In [ ]:
result = run_scenario(selected)
print('Run ID:', result['run_id'])
print('Agent sequence:', ' → '.join(result['agent_order']))

## 3. Inspect the agent trace

This is the execution sequence written to the Blackboard and audit stream.

In [ ]:
for event in result['events']:
    print(f"[{event['agent']:<22}] {event['message']}")

## 4. Inspect the shared Blackboard

Expand the structured payloads to see exactly what each agent contributed.

In [ ]:
for post in result['posts']:
    print('\n' + post['post_id'] + ' · ' + post['agent'])
    print(post['summary'])
    print(json.dumps(post['payload'], indent=2, ensure_ascii=False, default=str))

## 5. Final mitigation decision

The Coordinator only reads specialist posts. It does not query the CSV files directly or ask the language model to invent a result.

In [ ]:
print(json.dumps(result['decision'], indent=2, ensure_ascii=False, default=str))

## 6. Verify all three sample scenarios

Expected outcomes: supplier delay → REORDER, route block → REROUTE, stockout → REORDER.

In [ ]:
expected = {'supplier_delay': 'REORDER', 'route_block': 'REROUTE', 'stockout': 'REORDER'}
for scenario in scenarios:
    demo_result = run_scenario(dict(scenario))
    actual = demo_result['decision']['action']
    target = expected[scenario['disruption_type']]
    print(f"{scenario['scenario_id']}: {actual} {'✓' if actual == target else '✗ expected ' + target}")
    assert actual == target

## Submission artefacts

Each run is stored under `backend/runtime/runs/` as a JSON snapshot. Events are appended to `backend/runtime/append_only_run_log.jsonl`. Automated tests live in `backend/tests/test_scenarios.py`.